# Chapitre 5 — Nettoyage des données

**Durée estimée : 10-12 heures**

---

## Objectifs d'apprentissage

À la fin de ce chapitre, vous serez capable de :

1. **Appliquer** différentes stratégies de traitement des valeurs manquantes (suppression, imputation)
2. **Identifier et supprimer** les doublons en préservant les informations pertinentes
3. **Traiter** les valeurs aberrantes selon le contexte métier
4. **Nettoyer** les types de données (dates, numériques, texte) pour les rendre exploitables

---

## 5.2 Traitement des doublons

### Identification complète

In [2]:
import pandas as pd
import numpy as np

# Créer un DataFrame avec des doublons
df_doublons = pd.DataFrame({
    'id': [1, 2, 3, 4, 5, 1, 2],
    'nom': ['Alice', 'Bob', 'Charlie', 'David', 'Eve', 'Alice', 'Bob'],
    'email': ['a@t.com', 'b@t.com', 'c@t.com', 'd@t.com', 'e@t.com', 'a@t.com', 'b@t.com'],
    'montant': [100, 200, 150, 300, 250, 100, 200]
})

print("DataFrame avec doublons :")
print(df_doublons)

DataFrame avec doublons :
   id      nom    email  montant
0   1    Alice  a@t.com      100
1   2      Bob  b@t.com      200
2   3  Charlie  c@t.com      150
3   4    David  d@t.com      300
4   5      Eve  e@t.com      250
5   1    Alice  a@t.com      100
6   2      Bob  b@t.com      200


In [3]:
# Nombre de doublons exacts
print(f"Doublons exacts : {df_doublons.duplicated().sum()}")

# Voir toutes les lignes dupliquées (y compris les originaux)
doublons = df_doublons[df_doublons.duplicated(keep=False)]
print("\nToutes les lignes dupliquées :")
print(doublons.sort_values(by=['id']))

Doublons exacts : 2

Toutes les lignes dupliquées :
   id    nom    email  montant
0   1  Alice  a@t.com      100
5   1  Alice  a@t.com      100
1   2    Bob  b@t.com      200
6   2    Bob  b@t.com      200


In [4]:
# Doublons sur une clé spécifique
doublons_email = df_doublons[df_doublons.duplicated(subset=['email'], keep=False)]
print(f"Emails en double : {len(doublons_email)}")
print(doublons_email)

Emails en double : 4
   id    nom    email  montant
0   1  Alice  a@t.com      100
1   2    Bob  b@t.com      200
5   1  Alice  a@t.com      100
6   2    Bob  b@t.com      200


### Suppression des doublons

In [5]:
# Supprimer les doublons exacts (garder la première occurrence)
df_clean = df_doublons.drop_duplicates()
print("Après drop_duplicates() (keep='first' par défaut) :")
print(df_clean)

Après drop_duplicates() (keep='first' par défaut) :
   id      nom    email  montant
0   1    Alice  a@t.com      100
1   2      Bob  b@t.com      200
2   3  Charlie  c@t.com      150
3   4    David  d@t.com      300
4   5      Eve  e@t.com      250


In [6]:
# Garder la dernière occurrence
df_clean_last = df_doublons.drop_duplicates(keep='last')
print("Après drop_duplicates(keep='last') :")
print(df_clean_last)

Après drop_duplicates(keep='last') :
   id      nom    email  montant
2   3  Charlie  c@t.com      150
3   4    David  d@t.com      300
4   5      Eve  e@t.com      250
5   1    Alice  a@t.com      100
6   2      Bob  b@t.com      200


In [7]:
# Dé-dupliquer sur des colonnes spécifiques
df_clean_email = df_doublons.drop_duplicates(subset=['email'])
print("Après drop_duplicates(subset=['email']) :")
print(df_clean_email)

Après drop_duplicates(subset=['email']) :
   id      nom    email  montant
0   1    Alice  a@t.com      100
1   2      Bob  b@t.com      200
2   3  Charlie  c@t.com      150
3   4    David  d@t.com      300
4   5      Eve  e@t.com      250


In [8]:
# Dé-dupliquer en gardant la ligne avec le montant le plus élevé
df_doublons_v2 = pd.DataFrame({
    'client_id': [1, 2, 3, 1, 2],
    'nom': ['Alice', 'Bob', 'Charlie', 'Alice', 'Bob'],
    'montant': [100, 200, 150, 500, 50]  # Montants différents
})

print("Données avec montants différents :")
print(df_doublons_v2)

df_clean_max = df_doublons_v2.sort_values('montant', ascending=False).drop_duplicates(subset=['client_id'])
print("\nGarder le montant le plus élevé par client :")
print(df_clean_max)

Données avec montants différents :
   client_id      nom  montant
0          1    Alice      100
1          2      Bob      200
2          3  Charlie      150
3          1    Alice      500
4          2      Bob       50

Garder le montant le plus élevé par client :
   client_id      nom  montant
3          1    Alice      500
1          2      Bob      200
2          3  Charlie      150


### Gestion des quasi-doublons

In [9]:
# Quasi-doublons : même entité avec des variations mineures
df_quasi = pd.DataFrame({
    'id': [1, 2, 3, 4],
    'nom': ['Alice Martin', 'ALICE MARTIN', 'Bob Dupont', 'bob dupont'],
    'email': ['alice@test.com', 'ALICE@TEST.COM', 'bob@test.com', 'bob@test.com'],
    'date_maj': ['2024-01-01', '2024-03-15', '2024-02-01', '2024-04-10']
})

print("Données avec quasi-doublons :")
print(df_quasi)

Données avec quasi-doublons :
   id           nom           email    date_maj
0   1  Alice Martin  alice@test.com  2024-01-01
1   2  ALICE MARTIN  ALICE@TEST.COM  2024-03-15
2   3    Bob Dupont    bob@test.com  2024-02-01
3   4    bob dupont    bob@test.com  2024-04-10


In [10]:
# Normaliser avant de comparer
df_quasi['nom_normalise'] = df_quasi['nom'].str.lower().str.strip()
df_quasi['email_normalise'] = df_quasi['email'].str.lower().str.strip()

# Détecter les quasi-doublons
quasi_doublons = df_quasi[df_quasi.duplicated(subset=['nom_normalise', 'email_normalise'], keep=False)]
print(f"\nQuasi-doublons détectés : {len(quasi_doublons)}")
print(quasi_doublons)


Quasi-doublons détectés : 4
   id           nom           email    date_maj nom_normalise email_normalise
0   1  Alice Martin  alice@test.com  2024-01-01  alice martin  alice@test.com
1   2  ALICE MARTIN  ALICE@TEST.COM  2024-03-15  alice martin  alice@test.com
2   3    Bob Dupont    bob@test.com  2024-02-01    bob dupont    bob@test.com
3   4    bob dupont    bob@test.com  2024-04-10    bob dupont    bob@test.com


In [11]:
# Fusionner les quasi-doublons (garder le plus récent)
df_clean_quasi = df_quasi.sort_values('date_maj', ascending=False)
df_clean_quasi = df_clean_quasi.drop_duplicates(subset=['nom_normalise', 'email_normalise'], keep='first')
df_clean_quasi = df_clean_quasi.drop(columns=['nom_normalise', 'email_normalise'])

print("\nAprès fusion des quasi-doublons :")
print(df_clean_quasi)


Après fusion des quasi-doublons :
   id           nom           email    date_maj
3   4    bob dupont    bob@test.com  2024-04-10
1   2  ALICE MARTIN  ALICE@TEST.COM  2024-03-15


### ✍️ Exercice 5.3 : Dé-duplication intelligente (15 min)

In [12]:
import pandas as pd

# Données avec différents types de doublons
df_ex3 = pd.DataFrame({
    'id': [1, 2, 3, 4, 5, 6, 7, 8],
    'nom': ['Alice Martin', 'Bob Dupont', 'alice martin', 'Charlie Brown',
            'Bob Dupont', 'David Lee', 'ALICE MARTIN', 'Eve Wilson'],
    'email': ['alice@test.com', 'bob@test.com', 'alice@test.com', 'charlie@test.com',
              'bob@test.com', 'david@test.com', 'alice@test.com', 'eve@test.com'],
    'date_inscription': ['2024-01-15', '2024-01-16', '2024-02-01', '2024-01-17',
                         '2024-03-01', '2024-01-18', '2024-03-15', '2024-01-19'],
    'montant_total': [500, 1200, 300, 800, 1500, 600, 200, 900]
})

print("Données originales :")
print(df_ex3)

Données originales :
   id            nom             email date_inscription  montant_total
0   1   Alice Martin    alice@test.com       2024-01-15            500
1   2     Bob Dupont      bob@test.com       2024-01-16           1200
2   3   alice martin    alice@test.com       2024-02-01            300
3   4  Charlie Brown  charlie@test.com       2024-01-17            800
4   5     Bob Dupont      bob@test.com       2024-03-01           1500
5   6      David Lee    david@test.com       2024-01-18            600
6   7   ALICE MARTIN    alice@test.com       2024-03-15            200
7   8     Eve Wilson      eve@test.com       2024-01-19            900


In [13]:
# Étape 1 : Normaliser le nom et l'email
df_ex3['nom_norm'] = df_ex3['nom'].str.lower().str.strip()
df_ex3['email_norm'] = df_ex3['email'].str.lower().str.strip()

# Étape 2 : Identifier les doublons sur nom_norm + email_norm
doublons_ex3 = df_ex3[df_ex3.duplicated(subset=['nom_norm', 'email_norm'], keep=False)]
print(f"\nDoublons identifiés : {len(doublons_ex3)}")
print(doublons_ex3[['id', 'nom', 'email', 'montant_total']])


Doublons identifiés : 5
   id           nom           email  montant_total
0   1  Alice Martin  alice@test.com            500
1   2    Bob Dupont    bob@test.com           1200
2   3  alice martin  alice@test.com            300
4   5    Bob Dupont    bob@test.com           1500
6   7  ALICE MARTIN  alice@test.com            200


In [14]:
# Étape 3 : Garder l'enregistrement avec le montant_total le plus élevé
df_clean_ex3 = df_ex3.sort_values('montant_total', ascending=False)
df_clean_ex3 = df_clean_ex3.drop_duplicates(subset=['nom_norm', 'email_norm'], keep='first')

# Étape 4 : Nettoyer les colonnes temporaires
df_clean_ex3 = df_clean_ex3.drop(columns=['nom_norm', 'email_norm'])

print(f"\nRésultat : {len(df_clean_ex3)} lignes uniques")
print(df_clean_ex3)

# Question : Pourquoi avons-nous gardé la ligne avec le montant le plus élevé ?
print("\n→ On garde le client avec le plus de valeur (meilleur client)")


Résultat : 5 lignes uniques
   id            nom             email date_inscription  montant_total
4   5     Bob Dupont      bob@test.com       2024-03-01           1500
7   8     Eve Wilson      eve@test.com       2024-01-19            900
3   4  Charlie Brown  charlie@test.com       2024-01-17            800
5   6      David Lee    david@test.com       2024-01-18            600
0   1   Alice Martin    alice@test.com       2024-01-15            500

→ On garde le client avec le plus de valeur (meilleur client)
